# D39: FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Dạng D39: hai điều kiện mô-đun + liên hợp, đếm số số phức $z$ thỏa
$|z^2|=k|z-\bar z|$ và $|(z-c)(\bar z-ci)|=|z+ci|^2$. Phương pháp: rút
gọn điều kiện 2 trước (nhận ra $\bar z-ci=\overline{z+ci}$), tách 2
nhánh ($|z+ci|=0$ hoặc $|z-c|=|z+ci|$), nhánh 1 cho 0-1 nghiệm (kiểm tra
riêng), nhánh 2 luôn cho đúng 3 nghiệm, cộng lại. Bản chạy đầy đủ của
phương pháp đã kiểm chứng độc lập với **toàn bộ 91 câu D39** (khớp 91/91)
và người dùng đã tự test tay 10 câu (đa dạng cả 3 loại số) trong
`KLTN_D39_ReAct_Calculator_1cau.ipynb`.

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**) theo vòng
lặp ReAct thật:
`Thought → Action → Action Input → Observation → …`

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix);
model không còn quyền tự chọn viết văn xuôi mở đầu trước khi vào định dạng
ReAct.

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

Hạn mức tool gọi/câu (`30          # quy trinh day du ~13 luot tinh (k,c,z_th1,lhs1_th1,rhs1_th1,check,solve-b,b,z_th2,lhs1_th2,rhs1_th2,solve-a,tong) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap`) đủ dư cho khoảng 14-15 lượt tính
đầy đủ quy trình cộng tối đa 4 lượt so khớp phương án.

## Prompt & backend giống hệt bản 1 câu

Cell 3 (prompt + backend `MayTinh`, đặt trước khi `LLM(...)` khởi tạo để
an toàn với `multiprocessing.fork`) được **trích nguyên văn** từ notebook
1 câu bằng script `scratch/build_react_full_D39.py`; không gõ lại, nên
không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_D39.json` (90 câu, đã sinh sẵn từ
`Sinh_them_cau_hoi/So_phuc_day_du.csv`, đã loại câu gốc STT 66 nằm trong
few-shot) thành Kaggle Dataset (slug gợi ý `d39-full90`), gắn vào notebook,
bật GPU.

Kết quả: `/kaggle/working/d39_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d39-full90/plan_solve_prompts_D39.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d39_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'a', 'b'}  # a,b: toa do thuc/ao cua z tren duong thang trung truc (nhanh 2)

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D39_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision - this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True` - test all four, every time (see PART 4's matching step for why).
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo - retyping a small variation of the same idea will keep producing the same error. Stop, go back to PART 2's method for this exact situation (a formula giving `zoo`/an undefined result almost always means a special case described somewhere in PART 2 applies here - e.g. a division that is only valid when some quantity is nonzero), and use the alternative formula PART 2 gives for that case, typed as a normal exact expression - never invent a placeholder word like `undefined` as if it were a value; the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$ - there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change - so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).

20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15) - the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.


21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$ - turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Two conditions, two very different roles.** The first condition, $|z^2|=k|z-\overline{z}|$, is a single equation relating $z$ to itself - by itself it describes a whole curve of possible $z$, not a finite list. The second condition, $|(z-c)(\overline{z}-ci)|=|z+ci|^2$, at first glance seems to mix $z$ and $\overline{z}$ in a way that is hard to disentangle - but this is exactly the one worth simplifying FIRST, because doing so collapses it from "an equation mixing $z$ and $\overline{z}$" down to a small, explicit case split, each branch of which is far easier to combine with the first condition afterward.

**Recognizing $\overline{z}-ci$ in disguise.** Look at the second factor, $\overline{z}-ci$. By fact 21, this is exactly $\overline{z+ci}$ - the conjugate of the SAME quantity $z+ci$ that appears on the right-hand side. And by fact 4, $|\overline{w}|=|w|$ for any $w$, so $|\overline{z}-ci|=|\overline{z+ci}|=|z+ci|$. This single observation is the whole key to the problem: using fact 12 ($|uv|=|u||v|$), the second condition becomes
$$|z-c|\cdot|\overline{z}-ci| = |z+ci|^2 \iff |z-c|\cdot|z+ci| = |z+ci|^2,$$
an equation in $|z-c|$ and $|z+ci|$ only - no more separate, unresolved $\overline{z}$.

**From one equation to two branches.** Move everything to one side and factor out the common term $|z+ci|$:
$$|z+ci|\big(|z-c|-|z+ci|\big)=0.$$
A product of two real, nonnegative numbers is zero exactly when at least one factor is zero - so this single equation is EXACTLY equivalent to "$|z+ci|=0$ OR $|z-c|=|z+ci|$". These are two genuinely different situations (not two sub-cases of the same algebra), so each is worked out on its own, and every $z$ satisfying either one (and ALSO satisfying the first condition) counts toward the answer.

**Branch 1: $|z+ci|=0$ pins down a single concrete number.** A modulus is zero only when the number itself is zero, so $|z+ci|=0 \iff z=-ci$ - already fully concrete, no unknowns left. This branch never used the first condition, so $z=-ci$ is only an actual solution to the PROBLEM if it also satisfies $|z^2|=k|z-\overline{z}|$: compute $|z^2|$ and $k|z-\overline{z}|$ for this specific $z=-ci$ and compare them. If they are equal, branch 1 contributes exactly this one solution; if not, branch 1 contributes nothing at all - do not assume it always counts.

**Branch 2: $|z-c|=|z+ci|$ is itself a whole locus - describe it before touching the first condition.** This says $z$ is equidistant from two fixed points, $c$ (on the real axis) and $-ci$ (on the imaginary axis) (fact 17) - geometrically the perpendicular bisector of the segment between them, a full line, not a single point. To find WHICH points on that line also satisfy the first condition, write $z=a+bi$ ($a,b$ real) and square both sides: $(a-c)^2+b^2=a^2+(b+c)^2$. Both sides contain the cluster $a^2+b^2$ (fact 11) - solving this single equation for $b$ eliminates that cluster automatically and gives $b=-a$: the whole line collapses to "$b$ is always the negative of $a$", i.e. every point on it has the form $z=a-ai$ for some real $a$ still free to vary.

**Feeding $z=a-ai$ into the first condition turns the whole line into a short list of points.** With $z=a-ai$: $z^2=(a-ai)^2=-2a^2i$ (fact 2, using $i^2=-1$), so $|z^2|=2a^2$; and $z-\overline{z}=(a-ai)-(a+ai)=-2ai$, so $|z-\overline{z}|=2|a|$ (the absolute value matters - $a$ can be either sign). The first condition $|z^2|=k|z-\overline{z}|$ becomes $2a^2=2k|a|$. Solve this single equation for $a$ (do not simplify it by hand into $|a|(|a|-k)=0$ yourself - let the Calculator solve the equation directly, exactly as it is): it always has exactly three real solutions, $a=0$, $a=k$, and $a=-k$ (since $k$ here is always a stated positive coefficient, these three values are always distinct). Each distinct $a$ gives a distinct point $z=a-ai$ on the line, so branch 2 always contributes exactly three solutions: $z=0$, $z=k-ki$, $z=-k+ki$.

**Totalling - and checking the branches don't secretly share a solution.** The final count is (branch 1's contribution, 0 or 1) plus (branch 2's contribution, always 3). Before just adding, check whether branch 1's candidate $z=-ci$ could coincide with any of branch 2's three points: $z=-ci$ has real part $0$ and imaginary part $-c$; among branch 2's points, only $z=0$ also has real part $0$, but its imaginary part is $0$, not $-c$ - and $c\ne0$ in every version of this problem, so they can never be equal. The two branches never overlap, so the total is a plain sum, with no double-counting to worry about.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Problem: How many complex numbers $z$ satisfy $|z^2|=2|z-\overline{z}|$ and $|(z-4)(\overline{z}-4i)|=|z+4i|^2$?
A. $3$.   B. $1$.   C. $2$.   D. $4$.

Thought: First condition, quoted verbatim: "$|z^2|=2|z-\overline{z}|$" - the coefficient of $|z-\overline{z}|$ is $2$, so $k=2$. I store it.
Action: Calculator
Action Input: k = 2
Observation: k = 2

Thought: Second condition, quoted verbatim: "$|(z-4)(\overline{z}-4i)|=|z+4i|^2$" - matching against the template $(z-c)(\overline{z}-ci)=|z+ci|^2$: it shows "$z-4$", "$\overline{z}-4i$", "$z+4i$", every sign already matching the template as written, so $c=4$ directly (all three occurrences agree). I store it.
Action: Calculator
Action Input: c = 4
Observation: c = 4

Thought: Branch 1 candidate is $z=-ci$; I compute it.
Action: Calculator
Action Input: z_th1 = -c*I
Observation: z_th1 = -4*I

Thought: I check whether this candidate satisfies the first condition. First $|z^2|$.
Action: Calculator
Action Input: lhs1_th1 = Abs(z_th1**2)
Observation: lhs1_th1 = 16

Thought: Now $k|z-\overline{z}|$ for the same candidate.
Action: Calculator
Action Input: rhs1_th1 = k*Abs(z_th1 - conjugate(z_th1))
Observation: rhs1_th1 = 16

Thought: Compare the two sides to decide whether branch 1's candidate is a genuine solution.
Action: Calculator
Action Input: Eq(lhs1_th1, rhs1_th1)
Observation: True

Thought: Branch 1 is valid, contributing exactly 1 solution ($z=-4i$). Now branch 2: solve $(a-c)^2+b^2=a^2+(b+c)^2$ for $b$.
Action: Calculator
Action Input: solve(Eq((a-c)**2+b**2, a**2+(b+c)**2), b)
Observation: [-a]

Thought: So $b=-a$ on this line.
Action: Calculator
Action Input: b = -a
Observation: b = -a

Thought: Build the point on the line, $z=a+bi$.
Action: Calculator
Action Input: z_th2 = a + b*I
Observation: z_th2 = a - I*a

Thought: Compute $|z^2|$ for this point.
Action: Calculator
Action Input: lhs1_th2 = Abs(z_th2**2)
Observation: lhs1_th2 = 2*a**2

Thought: Compute $k|z-\overline{z}|$ for this point.
Action: Calculator
Action Input: rhs1_th2 = k*Abs(z_th2 - conjugate(z_th2))
Observation: rhs1_th2 = 4*Abs(a)

Thought: Solve the first condition for $a$ to find every point on the line that also satisfies it.
Action: Calculator
Action Input: solve(Eq(lhs1_th2, rhs1_th2), a)
Observation: [-2, 0, 2]

Thought: Three distinct values of $a$, so branch 2 contributes exactly 3 solutions. Branch 1 contributed 1, and I already know from PART 2's reasoning that branch 1's candidate ($z=-4i$, real part $0$, imaginary part $-4$) cannot coincide with any branch-2 point (only $z=0$ has real part $0$, and its imaginary part is $0\ne-4$), so no overlap to worry about. Total is a plain sum.
Action: Calculator
Action Input: tong = 1 + 3
Observation: tong = 4

Thought: Eq() here is a numeric check, not a symbolic proof, so I test ALL FOUR options every time, never stopping early. Read option A verbatim: "A. $3$."; A is 3. Test it.
Action: Calculator
Action Input: Eq(tong, 3)
Observation: False

Thought: Not A. Read option B verbatim: "B. $1$."; B is 1. Test it.
Action: Calculator
Action Input: Eq(tong, 1)
Observation: False

Thought: Not B. Read option C verbatim: "C. $2$."; C is 2. Test it.
Action: Calculator
Action Input: Eq(tong, 2)
Observation: False

Thought: Not C. Read option D verbatim: "D. $4$."; D is 4. Test it.
Action: Calculator
Action Input: Eq(tong, 4)
Observation: True

Thought: Exactly one option came back True: D. That is the answer.
Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: how many complex numbers $z$ satisfy $|z^2|=k|z-\overline{z}|$ AND $|(z-c)(\overline{z}-ci)|=|z+ci|^2$ (for specific constants $k,c$ given in this problem)? Follow these steps, using the Calculator for every computation.

1. **Finding $k$.** Quote the first condition verbatim from the problem (e.g. "$|z^2|=2|z-\overline{z}|$"), then read off the coefficient of $|z-\overline{z}|$ as $k$ (if no coefficient is written, $k=1$). Do not let any other number in the problem influence this - $k$ comes ONLY from this first condition.
2. **Finding $c$ - this is the step most likely to go wrong, so do it carefully.** Quote the second condition verbatim (e.g. "$|(z-4)(\overline{z}-4i)|=|z+4i|^2$" or "$|(z+3)(\overline{z}+3i)|=|z-3i|^2$"). It must match the template $(z-c)(\overline{z}-ci)=|z+ci|^2$ for SOME real number $c$ (possibly negative) - find that $c$ by matching signs, not by copying a digit: if the problem shows "$z-4$", "$\overline{z}-4i$", "$z+4i$" (every sign matches the template as written), then $c=4$. If instead it shows "$z+3$", "$\overline{z}+3i$", "$z-3i$" (every sign is the OPPOSITE of the template), then $c=-3$ (because $z+3=z-(-3)$). All three occurrences in the problem must agree with the SAME value of $c$ - if you find a value that only matches one or two of the three occurrences, you have the wrong sign; re-read and flip it.
3. Compute branch 1's candidate $z_{th1}=-ci$.
4. Check whether $z_{th1}$ satisfies the first condition: compute $|z_{th1}^2|$ and $k|z_{th1}-\overline{z_{th1}}|$ separately, then compare them with `Eq(...)`. Remember the exact result (branch 1 contributes 1 solution if they match, 0 if not) - do not assume either outcome in advance.
5. Solve $(a-c)^2+b^2=a^2+(b+c)^2$ for $b$ (declared free unknowns $a,b$) to get the line's equation in the form $b=\dots$, then store that as `b`.
6. Build $z_{th2}=a+bi$ using this $b$, then compute $|z_{th2}^2|$ and $k|z_{th2}-\overline{z_{th2}}|$.
7. Solve the equation "$|z_{th2}^2|$ equals $k|z_{th2}-\overline{z_{th2}}|$" for $a$. This always gives exactly 3 real values - branch 2 always contributes exactly 3 solutions, one for each value of $a$.
8. Add branch 1's contribution (from step 4) and branch 2's contribution (always 3) to get the total count. (The two branches never overlap - see PART 2's reasoning - so a plain sum is always correct; you do not need to re-derive this each time.)
9. For each option A, B, C, D in turn: read that option's value VERBATIM from the problem text (do not reuse a number from any earlier example), then compare with `Eq(total, <that option's value>)`. Test every option even after one already returns `True` - `Eq()` here is a numeric check, not a formal proof.
10. Write the final line `Final Answer: \boxed{<Letter>}` using whichever option matched.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 30          # quy trinh day du ~13 luot tinh (k,c,z_th1,lhs1_th1,rhs1_th1,check,solve-b,b,z_th2,lhs1_th2,rhs1_th2,solve-a,tong) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)